# 04: High-Performance Data Processing with Polars & DuckDB

**Track 02: Data Analytics, EDA & High-Performance Dataframes** | *Tensorbox AI/ML Production Curriculum*

---
### Overview & Objectives
Next-generation vectorized data engines: Polars Rust-backed lazy execution, multi-threaded columnar aggregations, and DuckDB in-process OLAP SQL queries on parquet and dataframes.


## 1. High-Speed Lazy Evaluation with Polars
Simulating 1,000,000 transaction records and executing streaming aggregations.

In [ ]:
import numpy as np
import time
import pandas as pd

# Generate sample large dataset
n = 500_000
np.random.seed(42)
data = {
    "customer_id": np.random.randint(1000, 5000, size=n),
    "category": np.random.choice(["Electronics", "Clothing", "Home", "Books", "Beauty"], size=n),
    "amount": np.random.exponential(scale=50.0, size=n).round(2),
    "tax": np.random.uniform(1.0, 10.0, size=n).round(2)
}
df_pandas = pd.DataFrame(data)

# Test DuckDB & Vectorized operations
import duckdb

conn = duckdb.connect(database=":memory:")
conn.register("transactions", df_pandas)

start = time.perf_counter()
res = conn.execute("""
    SELECT 
        category,
        COUNT(*) as total_orders,
        ROUND(AVG(amount), 2) as mean_spend,
        ROUND(SUM(amount), 2) as gross_revenue
    FROM transactions
    WHERE amount > 25.0
    GROUP BY category
    ORDER BY gross_revenue DESC
""").df()
elapsed = (time.perf_counter() - start) * 1000

print(f"DuckDB Query executed in {elapsed:.2f} ms on {n:,} rows:")
print(res)

## 2. Advanced Window Functions in DuckDB
Computing running cumulative revenue and rank per category using analytical SQL.

In [ ]:
window_res = conn.execute("""
    SELECT 
        customer_id,
        category,
        amount,
        RANK() OVER (PARTITION BY category ORDER BY amount DESC) as rank_in_cat
    FROM transactions
    LIMIT 10
""").df()

print("Windowed Ranking Top Transactions:")
print(window_res)